In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tbparse
%matplotlib inline

In [ ]:
final_csv_path = './result_csvs_and_pdfs_pp_on_alisim_2025_01_17/FINAL.csv'
final_df = pd.read_csv(final_csv_path)

In [ ]:
final_df['label'] = final_df[['model','train_data','test_data', 'param_id']].apply(lambda x:'_'.join(x), axis=1)
final_df['percent_epochs'] = final_df['train_epochs']/final_df['max_epochs']
final_df
final_df = final_df[final_df["test_data"].str.contains("alisim", na=False)]
final_df["test_num_leaves"] = final_df["test_data"].str.split("_").str[1].astype(int)
final_df
final_df["test_num_sites"] = final_df["test_data"].str.split("_sites").str[0].str.split("_").str[-1].astype(int)
final_df["test_num_alignments"] = final_df["test_data"].str.split("_alignments").str[0].str.split("_").str[-1].astype(int)
print(final_df)

final_df["train_num_leaves"] = final_df["train_data"].str.split("leaf").str[0].astype(int)
final_df["train_num_trees"] = final_df["train_data"].str.split("_").str[1].str.split("trees").str[0].astype(int)
final_df = final_df[final_df["train_num_leaves"] == final_df["test_num_leaves"]]

# final_df = final_df[final_df["train_data"].str.contains(r"^10leaf", na=False)]
# final_df = final_df[final_df["test_data"].str.contains(r"^10leaf", na=False)]
# print(final_df)

num_runs = len(final_df)
final_df

In [ ]:
final_df["model_and_train_data"] = final_df["model"] + final_df["train_data"]

# Sort models
model_order = ["TraverseMaxPooling", "TraverseAvgPooling", "TraverseNN"]
final_df['model'] = pd.Categorical(final_df['model'], categories=model_order, ordered=True)
df_sorted = final_df.sort_values(by=['model', 'train_num_leaves', 'train_num_trees', 'test_num_leaves'])
print(df_sorted)

In [ ]:
# heatmap of AUROCs for training on perfect phylogenies, testing on alisim generated data
final_df["model_and_train_data"] = f"{final_df['model']} {final_df['train_num_leaves']} leaves {final_df['train_num_trees']} trees"


plt.figure(figsize=(10,8))
# sns.set(rc={'figure.figsize': (3 * num_runs, 6)})
heatmap_data = df_sorted[["test_auroc", "test_num_sites", "test_num_alignments", "model", "train_num_trees"]]
heatmap_data.index=df_sorted["label"]
print(heatmap_data)
heatmap_data = df_sorted.pivot_table(
    index=['model', 'train_num_trees'],
    columns=['test_num_sites', 'test_num_leaves'],
    values='test_auroc'
)
# print(heatmap_data)
# sub_df = final_df[["model_and_train_data", "test_auroc"]]
# print(sub_df)
ax = sns.heatmap(
    heatmap_data,
    annot=True,
    cbar_kws={'label': 'Test AUROC'},
)
# ax.set_xlabel("Number of leaves in test data (100 trees)")
# ax.set_ylabel("Model - number of leaves - number of trees")
plt.tight_layout()
plt.savefig("./test_auroc_heatmap.pdf")
plt.show()

In [ ]:
# get dataframe from lightning log

def get_df_from_log(log_path):
    reader = tbparse.SummaryReader(log_path)
    return reader.scalars

In [ ]:
# get dfs from logs

hyperparam_dfs = {}
train_dfs = {}
test_dfs = {}
custom_dfs = {}

for index,row in final_df.iterrows():
    label_str = row.label
    label = (row.model,row.train_data,row.test_data,row.param_id)
    train_df = get_df_from_log(row.train_llog_path)
    train_dfs[label] = train_df
    test_df = get_df_from_log(row.test_llog_path)
    test_dfs[label] = test_df
    custom_df = get_df_from_log(row.train_clog_path)
    custom_dfs[label] = custom_df
    hyperparam_dfs[label] = {}
    for i in range(row.n_hyperparam_trials):
        hyperparam_dfs[label][i] = get_df_from_log(f'{row.hyperparam_llog_path}/version_{i}')


In [ ]:
models = ['model', 'train_data', 'test_data', 'param_id']
hyperparams = ['learning_rate', 'batch_size', 'accum_grad_batches', 'max_epochs', 'feature_length', 'dim_mlp_layers']

melt_df = pd.melt(final_df, id_vars='label', var_name='Metric', value_name='Value')
melt_df = melt_df[melt_df.Metric.isin(hyperparams)]
print(melt_df)

In [ ]:
# plot runtimes
runtime_df = final_df[['label', 'model_and_train_data', 'train_walltime', 'train_num_leaves', 'train_num_trees', 'model']]
# runtime_df = final_df.drop_duplicates(subset=['label', 'model_and_train_data', 'train_walltime', 'train_num_leaves', 'train_num_trees'])
runtime_df = runtime_df.sort_values(by=['train_num_leaves', 'train_num_trees'])
runtime_df["num_leaves_and_trees"] = runtime_df.apply(
    lambda row: f"{row['train_num_leaves']}leaves_{row['train_num_trees']}trees", axis=1
)

runtime_df["train_walltime_min"] = runtime_df["train_walltime"]/60
print(runtime_df["num_leaves_and_trees"])
for x_title,x_col in [["Training Runtime (in mins)","train_walltime_min"]]:
    sns.set(rc={'figure.figsize': (10, 10)})
    sns.barplot(y="num_leaves_and_trees", x=f"{x_col}", data=runtime_df, palette='deep', hue="model")
    plt.ylabel("(Model, Training Data, Test Data, Param Set)")
    plt.xlabel(f"{x_title}")
    plt.title(f"{x_title} by Model")
    plt.tight_layout()
    plt.savefig(f'result_csvs_and_pdfs_perfect_phylogenies_2025_01_13/{x_col}.barplot.pdf')
    plt.show()
    plt.clf()
print(runtime_df)